# Pengumpulan, Preprocessing, dan Pelabelan Sentimen Data X (Twitter)
### Penelitian Publikasi Ilmiah - Analisis Sentimen Usulan Reklasifikasi MSCI Indonesia

Notebook ini bertugas sebagai tahap persiapan data (*Data Preparation*):
1. **Crawling Data X:** Mengambil tweet menggunakan `tweet-harvest` untuk dua periode (Sebelum vs Sesudah).
2. **Data Preprocessing:** Cleaning, Case Folding, Tokenization, Stopword Removal, dan Stemming Sastrawi.
3. **Pelabelan Sentimen Lexicon-Based:** Menggunakan kamus InSet, Leksikon Pasar Modal, dan penanganan kata negasi.
4. **Penyimpanan Dataset Akhir:** Menyimpan data siap latih ke `dataset_msci_labeled.csv` untuk dipakai oleh notebook modeling (SVM dan Logistic Regression).

In [1]:
import sys
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8', errors='replace')

import os
import glob
import re
import time
import unicodedata
import urllib.request
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

os.makedirs('tweets-data', exist_ok=True)
os.makedirs('lexicon', exist_ok=True)
print("[OK] Inisialisasi lingkungan selesai!")

[OK] Inisialisasi lingkungan selesai!


---  
## 1. Pengumpulan Data X (Crawling via Tweet-Harvest)

In [ ]:
twitter_auth_token = 'a8b30e8aa375014d32e651b38165dcc88c0721f5'

queries = {
    "sebelum_1.csv": "MSCI Indonesia since:2026-10-01 until:2026-10-15",
    "sebelum_2.csv": "MSCI IHSG since:2026-10-01 until:2026-10-15",
    "sebelum_3.csv": "MSCI saham since:2026-10-01 until:2026-10-15",
    "sesudah_1.csv": "MSCI Indonesia since:2026-10-15 until:2026-10-27",
    "sesudah_2.csv": "MSCI IHSG since:2026-10-15 until:2026-10-27",
    "sesudah_3.csv": "MSCI standalone since:2026-10-15 until:2026-10-27",
    "sesudah_4.csv": "MSCI saham since:2026-10-15 until:2026-10-27"
}

for filename, search_query in queries.items():
    output_path = os.path.join("tweets-data", filename)
    if not os.path.exists(output_path):
        print(f"Mengambil data untuk: {search_query} -> {filename}")
        !npx -y tweet-harvest@2.7.1 -o "{filename}" -s "{search_query}" --tab "LATEST" -l 500 --token "{twitter_auth_token}"
    else:
        print(f"[SKIP] File '{filename}' sudah ada di folder tweets-data/")

print("\n[OK] Tahap 1 (Pengumpulan Data) Selesai!")

---  
## 2. Preprocessing Data (Cleaning, Case Folding, Tokenizing, Stopword, Stemming)

In [2]:
if os.path.exists('dataset_msci_preprocessed.csv'):
    print("[OK] Memuat data preprocessing yang sudah tersimpan...")
    df = pd.read_csv('dataset_msci_preprocessed.csv')
else:
    print("[INFO] Memproses data dari awal (tweets-data/*.csv)...")
    files = glob.glob("tweets-data/*.csv")
    list_df = []
    for f in files:
        d = pd.read_csv(f)
        fname = os.path.basename(f).lower()
        d['periode'] = 'Sebelum Pengumuman' if 'sebelum' in fname else 'Sesudah Pengumuman'
        list_df.append(d)
    df = pd.concat(list_df, ignore_index=True)
    
    df = df[~df['full_text'].astype(str).str.startswith('RT @')]
    df = df.drop_duplicates(subset=['full_text']).reset_index(drop=True)
    
    stemmer = StemmerFactory().create_stemmer()
    stopword_remover = StopWordRemoverFactory().create_stop_word_remover()
    
    def clean_text(text):
        text = str(text)
        text = re.sub(r'https?://\S+|www\.\S+', '', text)
        text = re.sub(r'[@#]\S+', '', text)
        text = ''.join(c for c in text if not unicodedata.category(c).startswith('S'))
        text = text.encode('ascii', 'ignore').decode('utf-8')
        text = re.sub(r'[^\w\s]', ' ', text)
        text = re.sub(r'\d+', ' ', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text
    
    print("Menjalankan Preprocessing...")
    df['clean_text'] = df['full_text'].apply(clean_text)
    df['case_folding'] = df['clean_text'].str.lower()
    df['tokens'] = df['case_folding'].apply(lambda x: x.split())
    df['stopword_removed'] = df['case_folding'].apply(lambda x: stopword_remover.remove(x))
    
    all_words = set(w for text in df['stopword_removed'] for w in text.split())
    t0 = time.time()
    stem_dict = {w: stemmer.stem(w) for w in all_words}
    df['stemmed_text'] = df['stopword_removed'].apply(lambda text: " ".join(stem_dict.get(w, w) for w in text.split()))
    print(f"Stemming selesai dalam {time.time()-t0:.1f} detik!")
    
    df.to_csv('dataset_msci_preprocessed.csv', index=False, encoding='utf-8-sig')

print(f"[OK] Data preprocessing siap: {len(df)} tweet")
display(df[['full_text', 'stemmed_text']].head(2))

[OK] Memuat data preprocessing yang sudah tersimpan...
[OK] Data preprocessing siap: 1039 tweet


,full_text,stemmed_text
0,Sentimen MSCI menjadi salah satu faktor yang t...,sentimen msci jadi salah satu faktor turut bay...
1,penjelsan benix soal kenapa ihsg jatuh\n\nihsg...,penjelsan benix soal ihsg jatuh ihsg pernah se...


---  
## 3. Pelabelan Sentimen (Lexicon-Based: InSet + Market Lexicon)

In [3]:
# Safety check jika DataFrame df belum termuat di kernel sesi ini
if 'df' not in locals():
    if os.path.exists('dataset_msci_preprocessed.csv'):
        df = pd.read_csv('dataset_msci_preprocessed.csv')
    else:
        raise FileNotFoundError("File 'dataset_msci_preprocessed.csv' belum ditemukan. Jalankan sel preprocessing terlebih dahulu.")

# Buat direktori penampung lexicon jika belum ada
os.makedirs('lexicon', exist_ok=True)

url_pos = 'https://raw.githubusercontent.com/fajri91/InSet/master/positive.tsv'
url_neg = 'https://raw.githubusercontent.com/fajri91/InSet/master/negative.tsv'
path_pos = 'lexicon/positive.tsv'
path_neg = 'lexicon/negative.tsv'

if not os.path.exists(path_pos): 
    urllib.request.urlretrieve(url_pos, path_pos)
if not os.path.exists(path_neg): 
    urllib.request.urlretrieve(url_neg, path_neg)

lexicon = {}
with open(path_pos, 'r', encoding='utf-8') as f:
    for line in f:
        p = line.strip().split('\t')
        if len(p) >= 2:
            try: 
                lexicon[p[0].lower()] = float(p[1])
            except ValueError: 
                pass

with open(path_neg, 'r', encoding='utf-8') as f:
    for line in f:
        p = line.strip().split('\t')
        if len(p) >= 2:
            try: 
                lexicon[p[0].lower()] = float(p[1])
            except ValueError: 
                pass

# Tambahan kamus pasar modal
market_lexicon = {
    'cuan': 4.0, 'rebound': 3.5, 'bullish': 3.5, 'menguat': 3.0, 'bangkit': 3.0,
    'terbang': 3.0, 'hijau': 2.5, 'akumulasi': 2.5, 'untung': 3.0, 'profit': 3.5,
    'inflow': 2.5, 'net buy': 3.0, 'optimis': 3.0, 'potensi': 2.0, 'tumbuh': 2.5,
    'surplus': 2.5, 'dividen': 2.5, 'meroket': 4.0,
    'anjlok': -4.0, 'ambles': -4.0, 'terjun': -3.5, 'bearish': -3.5, 'melemah': -3.0,
    'terpuruk': -3.5, 'tertekan': -3.0, 'merah': -2.0, 'loss': -3.0, 'cutloss': -3.5,
    'panik': -3.5, 'panic': -3.5, 'outflow': -2.5, 'net sell': -3.0, 'banting': -3.0,
    'hancur': -4.0, 'terhantam': -3.5, 'dihantam': -3.5, 'diancam': -3.0, 'ancam': -3.0,
    'momok': -3.5, 'membayangi': -2.5, 'gagal': -3.0, 'rugi': -3.5, 'koreksi': -2.0,
    'jebol': -3.0, 'rusak': -3.0
}
lexicon.update(market_lexicon)
negation_words = {'tidak', 'bukan', 'tak', 'tiada', 'ga', 'gak', 'ngga', 'nggak', 'jangan', 'belum'}

def analyze_sentiment(text):
    words = str(text).split()
    score = 0.0
    negate = False
    for word in words:
        if word in negation_words:
            negate = True
            continue
        weight = lexicon.get(word, 0.0)
        if weight != 0.0:
            score += -weight * 0.75 if negate else weight
            negate = False
        else:
            negate = False
            
    if score > 0.5: 
        return 'Positif', round(score, 2)
    elif score < -0.5: 
        return 'Negatif', round(score, 2)
    else: 
        return 'Netral', round(score, 2)

target_text = df['case_folding'] if 'case_folding' in df.columns else df['clean_text']
sent_results = [analyze_sentiment(t) for t in target_text]
df['sentimen'] = [r[0] for r in sent_results]
df['skor_sentimen'] = [r[1] for r in sent_results]
df.to_csv('dataset_msci_labeled.csv', index=False, encoding='utf-8-sig')

print("=== Distribusi Sentimen Hasil Pelabelan ===")
print(df['sentimen'].value_counts())
print("\nPersentase Sentimen (%):")
print((df['sentimen'].value_counts(normalize=True) * 100).round(2))

crosstab_res = pd.crosstab(df['periode'], df['sentimen'], margins=True, margins_name='Total')
display(crosstab_res)

print("\n[OK] Dataset berlabel berhasil disimpan ke 'dataset_msci_labeled.csv'.")
print("[OK] Siap diproses untuk pemodelan di notebook SVM dan Logistic Regression!")

=== Distribusi Sentimen Hasil Pelabelan ===
sentimen
Negatif    809
Positif    115
Netral     115
Name: count, dtype: int64

Persentase Sentimen (%):
sentimen
Negatif    77.86
Positif    11.07
Netral     11.07
Name: proportion, dtype: float64


sentimen,Negatif,Netral,Positif,Total
periode,,,,
Sebelum Pengumuman,54,6,8,68
Sesudah Pengumuman,755,109,107,971
Total,809,115,115,1039



[OK] Dataset berlabel berhasil disimpan ke 'dataset_msci_labeled.csv'.
[OK] Siap diproses untuk pemodelan di notebook SVM dan Logistic Regression!
